In [19]:
# Streaming helpers for ChatGPT export

import os
from pathlib import Path

import ijson
from mem0 import MemoryClient


def iter_conversations(path=Path("conversations.json")):
    # conversations.json is a top-level JSON array
    with path.open("rb") as f:
        for convo in ijson.items(f, "item"):
            yield convo


def extract_messages(convo):
    # Follow the parent chain from current_node back to root, then reverse
    mapping = convo.get("mapping") or {}
    node_id = convo.get("current_node")
    if not mapping or not node_id or node_id not in mapping:
        return []

    chain = []
    seen = set()
    while node_id and node_id in mapping and node_id not in seen:
        seen.add(node_id)
        node = mapping[node_id] or {}
        chain.append(node)
        node_id = node.get("parent")

    messages = []
    for node in reversed(chain):
        msg = node.get("message")
        if not isinstance(msg, dict):
            continue

        role = (msg.get("author") or {}).get("role")
        if role not in {"user", "assistant", "system"}:
            continue

        content = msg.get("content") or {}
        if content.get("content_type") != "text":
            continue

        parts = content.get("parts") or []
        text = "\n".join(p for p in parts if isinstance(p, str)).strip()
        if not text:
            continue

        # message/create_time in the export is usually a unix timestamp (float seconds)
        messages.append(
            {"role": role, "content": text, "create_time": msg.get("create_time")}
        )

    return messages


def select_conversation(
    path=Path("conversations.json"), convo_id=None, title_contains=None, index=None
):
    # Pick exactly one conversation using id/title/index (in that priority order)
    title_contains_lc = title_contains.lower() if title_contains else None

    for idx, convo in enumerate(iter_conversations(path)):
        if convo_id and convo.get("id") == convo_id:
            return convo

        if (
            title_contains_lc
            and title_contains_lc in (convo.get("title") or "").lower()
        ):
            return convo

        if index is not None and idx == index:
            return convo

    return None


In [ ]:
# Peek at the first few conversations

preview = []
for idx, convo in enumerate(iter_conversations()):
    preview.append({"idx": idx, "id": convo.get("id"), "title": convo.get("title")})
    if idx >= 4:
        break

preview

[{'idx': 0,
  'id': '69684f48-df2c-8333-a3f6-af7ae6daca98',
  'title': 'Project Context Export'},
 {'idx': 1,
  'id': '69684f31-0f78-832b-bc61-56b870733ab5',
  'title': 'Project Context Export'},
 {'idx': 2,
  'id': '6968460c-fec0-8325-879f-46c0dc859ebf',
  'title': 'Memory Update and Migration'},
 {'idx': 3, 'id': '69684389-5dbc-8332-a44a-9cb3c9305fa6', 'title': 'New chat'},
 {'idx': 4,
  'id': '696843fa-4390-832c-a637-e9ed0884bd87',
  'title': 'Devs Watch Discussion'}]

In [ ]:
# Pick one conversation and extract a small message slice

TARGET_ID = None  # e.g. "6968460c-fec0-8325-879f-46c0dc859ebf"
TITLE_CONTAINS = "Memory Update"  # set to None to disable
TARGET_INDEX = None  # set to an int to use index selection

selected = select_conversation(
    convo_id=TARGET_ID, title_contains=TITLE_CONTAINS, index=TARGET_INDEX
)
if not selected:
    raise RuntimeError("No conversation matched")

messages = extract_messages(selected)
window = [{"role": m["role"], "content": m["content"]} for m in messages[:12]]

{
    "conversation_id": selected.get("id"),
    "title": selected.get("title"),
    "total_messages_extracted": len(messages),
    "window_preview": window,
}

{'conversation_id': '6968460c-fec0-8325-879f-46c0dc859ebf',
 'title': 'Memory Update and Migration',
 'total_messages_extracted': 19,
 'window_preview': [{'role': 'user',
   'content': 'please review these memories and conversation histories from claude - im migrating everything back over. update our memories here when relevant. note that not everything claude captured is high signal - my job doesnt define me, that "current work" list is woefully incomplete, a lot of the "stated facts" are really more nebulous than claude makes it seem. use your discretion - or even better, ask me to clarify signal (using easy to answer metrics/questions...)\n\n=======\n\n# Harrison - Complete Memory Debrief\n\nGenerated: January 14, 2026\n\n---\n\n## Identity & Background\n\n**Name:** Harrison\n\n**Location:** Norfolk, Virginia (Eastern Time)\n\n**Professional Background:**\n- Army officer who transitioned into applied AI/ML work\n- Master\'s in Computer Science from Georgia Tech\n- Currently an AI en

In [ ]:
# Bulk processing helpers (streaming)


def chunked(items, chunk_messages):
    # chunk_messages = number of messages in each Mem0 add() call
    for i in range(0, len(items), chunk_messages):
        yield items[i : i + chunk_messages]


def chunk_time_range(messages):
    times = [m.get("create_time") for m in messages if m.get("create_time") is not None]
    if not times:
        return None, None
    return min(times), max(times)


def import_conversations(
    *,
    export_path="conversations.json",
    user_id="harrison",
    api_key=None,
    dry_run=True,
    start_index=0,
    max_convos=None,
    convo_id=None,
    title_contains=None,
    chunk_messages=12,
    titles_limit=200,
):
    # Streams the export and imports in small message chunks
    path = Path(export_path)

    if api_key is None:
        api_key = os.environ.get("MEM0_API_KEY")

    client = None
    if not dry_run:
        client = MemoryClient(api_key=api_key)

    chunks_processed = 0
    convos_imported = 0
    processed = []

    for idx, convo in enumerate(iter_conversations(path)):
        if idx < start_index:
            continue

        if convo_id and convo.get("id") != convo_id:
            continue

        if (
            title_contains
            and title_contains.lower() not in (convo.get("title") or "").lower()
        ):
            continue

        messages = extract_messages(convo)
        if not messages:
            continue

        run_id = convo.get("id") or f"chatgpt-{idx}"

        convo_msg_start, convo_msg_end = chunk_time_range(messages)
        base_metadata = {
            "source": "chatgpt_export",
            "title": convo.get("title"),
            "conversation_id": convo.get("id"),
            "index": idx,
            "convo_create_time": convo.get("create_time"),
            "convo_update_time": convo.get("update_time"),
            "convo_msg_start_time": convo_msg_start,
            "convo_msg_end_time": convo_msg_end,
        }

        chunk_idx = 0
        for chunk in chunked(messages, chunk_messages):
            chunk_start, chunk_end = chunk_time_range(chunk)

            metadata = dict(base_metadata)
            metadata["chunk_idx"] = chunk_idx
            metadata["chunk_start_time"] = chunk_start
            metadata["chunk_end_time"] = chunk_end

            if not dry_run:
                mem0_messages = [
                    {"role": m["role"], "content": m["content"]} for m in chunk
                ]
                client.add(
                    messages=mem0_messages,
                    user_id=user_id,
                    run_id=run_id,
                    metadata=metadata,
                    infer=True,
                )

            chunks_processed += 1
            chunk_idx += 1

        convos_imported += 1
        if len(processed) < titles_limit:
            processed.append(
                {"idx": idx, "id": convo.get("id"), "title": convo.get("title")}
            )

        if max_convos is not None and convos_imported >= max_convos:
            break

    return {
        "dry_run": dry_run,
        "convos_imported": convos_imported,
        "chunks_processed": chunks_processed,
        "processed": processed,
    }


In [22]:
# Example: do a dry-run first, then flip dry_run=False

result = import_conversations(
    dry_run=True,
    start_index=0,
    max_convos=5,
    chunk_messages=12,
    title_contains=None,
    convo_id=None,
)

result

{'dry_run': True,
 'convos_imported': 5,
 'chunks_processed': 6,
 'processed': [{'idx': 0,
   'id': '69684f48-df2c-8333-a3f6-af7ae6daca98',
   'title': 'Project Context Export'},
  {'idx': 1,
   'id': '69684f31-0f78-832b-bc61-56b870733ab5',
   'title': 'Project Context Export'},
  {'idx': 2,
   'id': '6968460c-fec0-8325-879f-46c0dc859ebf',
   'title': 'Memory Update and Migration'},
  {'idx': 3,
   'id': '69684389-5dbc-8332-a44a-9cb3c9305fa6',
   'title': 'New chat'},
  {'idx': 4,
   'id': '696843fa-4390-832c-a637-e9ed0884bd87',
   'title': 'Devs Watch Discussion'}]}